# 03 · Cross-encoder: baseline B3 (Amazon MiniLM reproduction)

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-12-v2` on query + product title with MSE on the gains, using Amazon's settings, and scores val and test. The same notebook is the starting point for Direction 1 (input fields, DeBERTa) and Direction 2 (losses).

**Kaggle settings:** Accelerator **GPU T4** (or P100), **Internet on**.

**Inputs:**
- Add Input → Notebook Output → **preprocess notebook** (`pairs.parquet`, `products_clean.parquet`). Required.
- Add Input → Notebook Output → **bm25 notebook**. Optional: enables the paired comparison with BM25.

**How to run:**
1. `MODE = 'smoke'` (5% of train queries, 1 seed, plus the GPU speed benchmark). Run interactively, about 10–15 minutes. Check the smoke-test verdict at the end of section 4.
2. `MODE = 'full'` (all train queries, seeds 42 and 43), then **Save Version → Save & Run All**. Estimated under 1 hour; the training log prints the real speed.

**Settings from Amazon's `ranking/train.py`:** MiniLM-L12 (MS MARCO), `num_labels = 1` with no activation, MSE on gains E 1 / S 0.1 / C 0.01 / I 0, title only, batch 32, 1 epoch, lr 7e-6, 5,000 warm-up steps with linear decay, AdamW (weight decay 0.01, gradient clipping 1.0).

**Differences from Amazon:**
- `max_len` is 128 instead of 512. Query + title almost never reaches 128 tokens; section 2 prints the truncated share.
- The model trains on our train split (18,799 queries). Amazon trained on the official train minus 400 queries, about 10% more data.
- fp16 mixed precision, for speed on T4.

**Output** (`/kaggle/working/results/`): `<run>.json` and `scores/<run>_{val,test}.parquet`, in the same format as `02_bm25`, plus `gpu_benchmark.json` in smoke mode.

In [4]:
import gc, glob, json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train queries, seed 42, + GPU benchmark | 'full': all train queries, seeds 42 and 43

CFG = {
    'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'fields': 'title',        # 'title' | 'title_bc' | 'main' (Direction 1)
    'max_len': 128,
    'loss': 'mse',            # Direction 2 adds 'ce4' and 'listnet'
    'epochs': 1,
    'lr': 7e-6,
    'batch_size': 32,
    'warmup_steps': 5000,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'fp16': True,
}
SEEDS = [42] if MODE == 'smoke' else [42, 43]
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0    # share of train queries
EVAL_EVERY = 100 if MODE == 'smoke' else 2000   # steps between val checks (monitoring only, no early stopping)
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']
BENCHMARK = MODE == 'smoke'
SAVE_MODEL = False

SEED = 42
TIE_SEEDS = (0, 1, 2)
GAINS = {'official': {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0},
         'amazon_script': {'E': 1.0, 'S': 0.01, 'C': 0.1, 'I': 0.0}}
OUT = Path('/kaggle/working')
RES = OUT / 'results'
(RES / 'scores').mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
DEVICE = 'cuda'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| seeds', SEEDS, '| subsample', SUBSAMPLE)

torch 2.11.0+cu130 | transformers 5.20.2 | Tesla T4
MODE smoke | seeds [42] | subsample 0.05


## 1. Evaluation functions

Copied unchanged from `02_bm25`, so both notebooks score runs the same way.

In [5]:
def _positions(codes):
    # 0-based rank inside each query; `codes` must be sorted so every query is contiguous
    n = len(codes)
    start = np.r_[0, np.flatnonzero(np.diff(codes)) + 1]
    return np.arange(n) - np.repeat(start, np.diff(np.r_[start, n]))


def query_metrics(df, score, seeds=TIE_SEEDS):
    # Per-query metrics of one score vector. df needs query_id and esci_label, in the same row order as score.
    score = np.asarray(score, dtype=float)
    assert len(score) == len(df) and np.isfinite(score).all()
    codes, qids = pd.factorize(df['query_id'].to_numpy())
    nq = len(qids)
    lab = df['esci_label'].to_numpy()
    gain = {k: pd.Series(lab).map(m).to_numpy() for k, m in GAINS.items()}
    is_e = lab == 'E'

    def dcg(order, g, k=None):
        c = codes[order]
        pos = _positions(c)
        w = 1 / np.log2(pos + 2)
        if k is not None:
            w = np.where(pos < k, w, 0.0)
        return np.bincount(c, g[order] * w, minlength=nq)

    ideal = {k: np.lexsort((-g, codes)) for k, g in gain.items()}
    idcg = {k: dcg(ideal[k], g) for k, g in gain.items()}
    idcg10 = dcg(ideal['official'], gain['official'], 10)

    acc = {'ndcg': 0.0, 'ndcg_amazon': 0.0, 'ndcg10': 0.0, 'mrr_e': 0.0}
    for sd in seeds:
        order = np.lexsort((np.random.default_rng(sd).random(len(df)), -score, codes))   # query, score desc, random
        acc['ndcg'] = acc['ndcg'] + dcg(order, gain['official'])
        acc['ndcg_amazon'] = acc['ndcg_amazon'] + dcg(order, gain['amazon_script'])
        acc['ndcg10'] = acc['ndcg10'] + dcg(order, gain['official'], 10)
        c, pos, e = codes[order], _positions(codes[order]), is_e[order]
        first_e = np.full(nq, np.inf)
        np.minimum.at(first_e, c[e], pos[e])
        acc['mrr_e'] = acc['mrr_e'] + 1 / (first_e + 1)

    n = len(seeds)
    with np.errstate(divide='ignore', invalid='ignore'):   # IDCG = 0 (all-I query) -> NaN, skipped in means
        return pd.DataFrame({
            'query_id': qids,
            'n_cand': np.bincount(codes, minlength=nq),
            'ndcg': acc['ndcg'] / n / idcg['official'],
            'ndcg_amazon': acc['ndcg_amazon'] / n / idcg['amazon_script'],
            'ndcg10': acc['ndcg10'] / n / idcg10,
            'mrr_e': np.where(np.bincount(codes, is_e.astype(float), minlength=nq) > 0, acc['mrr_e'] / n, np.nan),
        })


def e_over_s(df, score):
    # Share of (E, S) pairs inside a query where E scores higher; ties count 0.5 (Mann-Whitney U), pooled.
    f = pd.DataFrame({'q': df['query_id'].to_numpy(), 'l': df['esci_label'].to_numpy(),
                      's': np.asarray(score, dtype=float)})
    f = f[f['l'].isin(['E', 'S'])].copy()
    f['rk'] = f.groupby('q')['s'].rank(method='average')
    f['is_e'] = f['l'] == 'E'
    agg = f.groupby('q').agg(n_e=('is_e', 'sum'), n=('is_e', 'size'))
    agg['r_e'] = f[f['is_e']].groupby('q')['rk'].sum().reindex(agg.index).fillna(0.0)
    agg['n_s'] = agg['n'] - agg['n_e']
    agg = agg[(agg['n_e'] > 0) & (agg['n_s'] > 0)]
    u = agg['r_e'] - agg['n_e'] * (agg['n_e'] + 1) / 2
    return float(u.sum() / (agg['n_e'] * agg['n_s']).sum())


def bootstrap_ci(x, n_boot=1000, seed=SEED):
    x = np.asarray(x, dtype=float)
    means = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)]


def paired_bootstrap(pq_a, pq_b, metric='ndcg', n_boot=1000, seed=SEED):
    # Mean of (b - a) over the same queries, its 95% CI, and the share of resamples where b <= a.
    d = pq_a[['query_id', metric]].merge(pq_b[['query_id', metric]], on='query_id', suffixes=('_a', '_b')).dropna()
    diff = (d[metric + '_b'] - d[metric + '_a']).to_numpy()
    means = diff[np.random.default_rng(seed).integers(0, len(diff), (n_boot, len(diff)))].mean(axis=1)
    return {'diff': round(float(diff.mean()), 4),
            'ci95': [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)],
            'p_b_le_a': float((means <= 0).mean())}


METRICS = ['ndcg', 'ndcg_amazon', 'ndcg10', 'mrr_e']


def summarize(df, score):
    pq = query_metrics(df, score)
    s = {m: round(float(pq[m].mean()), 4) for m in METRICS}
    s['ndcg_ci95'] = bootstrap_ci(pq['ndcg'].dropna())
    s['e_over_s'] = round(e_over_s(df, score), 4)
    s['n_queries'] = int(pq['ndcg'].notna().sum())
    s['n_skipped_all_irrelevant'] = int(pq['ndcg'].isna().sum())
    return s, pq

## 2. Data

In [6]:
def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input: Add Input -> Notebook Output -> preprocess notebook'
    return Path(hits[0])


TEXT_COL = {'title': 'title', 'title_bc': 'text_title_bc', 'main': 'text_main'}

pairs = pd.read_parquet(find('pairs.parquet'),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'gain', 'split', 'fold10'])
prod = pd.read_parquet(find('products_clean.parquet'), columns=['product_id', TEXT_COL[CFG['fields']], 'text_main'])
pairs['text'] = pairs['product_id'].map(pd.Series(prod[TEXT_COL[CFG['fields']]].to_numpy(), index=prod['product_id']))
MAIN_TEXT = pd.Series(prod['text_main'].to_numpy(), index=prod['product_id'])   # only for the benchmark
del prod
assert pairs['text'].notna().all()

MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_all = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_all['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train = train_all[train_all['query_id'].isin(keep)].reset_index(drop=True)
else:
    train = train_all.reset_index(drop=True)
print(f'train {train["query_id"].nunique():,} queries / {len(train):,} pairs | '
      f'val {MASK["val"].sum():,} pairs | test {MASK["test"].sum():,} pairs | fields = {CFG["fields"]}')

tok = AutoTokenizer.from_pretrained(CFG['model'])
smp = pairs.sample(20_000, random_state=SEED)
n_tok = np.array([len(x) for x in tok(smp['query'].tolist(), smp['text'].tolist())['input_ids']])
print(f'query + text tokens: median {np.median(n_tok):.0f}, p99 {np.percentile(n_tok, 99):.0f}, '
      f'max {n_tok.max()} | truncated at max_len {CFG["max_len"]}: {(n_tok > CFG["max_len"]).mean():.2%}')

train 940 queries / 18,674 pairs | val 41,513 pairs | test 181,701 pairs | fields = title
query + text tokens: median 33, p99 70, max 126 | truncated at max_len 128: 0.00%


## 3. Training and prediction

- Plain PyTorch loop with the Hugging Face model (`num_labels = 1`, raw logit as the score), so Direction 2 only needs a new loss.
- Batches are shuffled pairs (seeded). Pairs are tokenized on the fly and padded to the longest pair in the batch.
- Prediction sorts pairs by length to reduce padding, then puts the scores back in the original order.
- Every `EVAL_EVERY` steps the full val set is scored, only to watch the curve. The model kept is always the one after the last step, as in Amazon's script.

In [17]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def encode(tokenizer, queries, texts, max_len):
    return tokenizer(list(queries), list(texts), truncation=True, max_length=max_len,
                     padding=True, return_tensors='pt').to(DEVICE)


def make_optimizer(model, cfg):
    no_decay = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         'weight_decay': cfg['weight_decay']},
        {'params': [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(groups, lr=cfg['lr'])


def compute_loss(pred, target, cfg):
    if cfg['loss'] == 'mse':
        return F.mse_loss(pred.float(), target)
    raise ValueError(f'unknown loss {cfg["loss"]}')


@torch.no_grad()
def predict(model, tokenizer, df, max_len, batch_size=512):
    model.eval()
    order = np.argsort((df['query'].str.len() + df['text'].str.len()).to_numpy(), kind='stable')
    q, t = df['query'].to_numpy()[order], df['text'].to_numpy()[order]
    out = np.empty(len(df), dtype=np.float32)
    for i in range(0, len(df), batch_size):
        with torch.autocast('cuda', dtype=torch.float16, enabled=CFG['fp16']):
            logits = model(**encode(tokenizer, q[i:i + batch_size], t[i:i + batch_size], max_len)).logits
        out[i:i + batch_size] = logits.squeeze(-1).float().cpu().numpy()
    model.train()
    scores = np.empty_like(out)
    scores[order] = out
    return scores


def train_ce(cfg, train_df, seed, eval_df=None, eval_every=None):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(cfg['model'])
    model = AutoModelForSequenceClassification.from_pretrained(cfg['model'], num_labels=1).float().to(DEVICE)
    q, t = train_df['query'].to_numpy(), train_df['text'].to_numpy()
    y = train_df['gain'].to_numpy(np.float32)
    loader = DataLoader(np.arange(len(train_df)), batch_size=cfg['batch_size'], shuffle=True, drop_last=True,
                        generator=torch.Generator().manual_seed(seed))
    total = len(loader) * cfg['epochs']
    warmup = min(cfg['warmup_steps'], int(0.42 * total))   # Amazon: 5,000 of ~11.8k steps; scaled down for smoke runs
    opt = make_optimizer(model, cfg)
    sched = get_linear_schedule_with_warmup(opt, warmup, total)
    scaler = make_scaler(cfg['fp16'])

    loss_log, val_curve = [], []
    step, run_loss, t0 = 0, torch.zeros((), device=DEVICE), time.time()
    model.train()
    for _ in range(cfg['epochs']):
        for idx in loader:
            idx = idx.numpy()
            with torch.autocast('cuda', dtype=torch.float16, enabled=cfg['fp16']):
                pred = model(**encode(tokenizer, q[idx], t[idx], cfg['max_len'])).logits.squeeze(-1)
            loss = compute_loss(pred, torch.from_numpy(y[idx]).to(DEVICE), cfg)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            run_loss += loss.detach()
            if step % 100 == 0 or step == total:
                n = 100 if step % 100 == 0 else step % 100
                avg = run_loss.item() / n
                assert np.isfinite(avg), f'loss is not finite at step {step}'
                loss_log.append({'step': step, 'loss': round(avg, 6), 'lr': sched.get_last_lr()[0]})
                run_loss.zero_()
                if step % 1000 == 0 or step == total or MODE == 'smoke':
                    el = time.time() - t0
                    print(f'step {step:>6}/{total} | loss {avg:.5f} | {step / el:.1f} it/s | '
                          f'{el / 60:.1f} min, ~{(total - step) / (step / el) / 60:.0f} min left')
            if eval_df is not None and eval_every and (step % eval_every == 0 or step == total):
                m, _ = summarize(eval_df, predict(model, tokenizer, eval_df, cfg['max_len']))
                val_curve.append({'step': step, 'val_ndcg': m['ndcg'], 'val_e_over_s': m['e_over_s']})
                print(f'   val @ {step}: nDCG {m["ndcg"]} | E-over-S {m["e_over_s"]}')
    info = {'steps': step, 'warmup': warmup, 'train_minutes': round((time.time() - t0) / 60, 1),
            'it_per_s': round(step / (time.time() - t0), 2), 'loss_log': loss_log, 'val_curve': val_curve}
    return model, tokenizer, info

## 4. Logging runs, zero-shot model and B3

`log_run` writes the same files as in `02_bm25`. If the bm25 notebook output is attached, every run is also compared with `b1_bm25_main` by paired bootstrap.

The **zero-shot** run scores pairs with the MS MARCO model before any fine-tuning. It is the starting point of B3 and shows how much fine-tuning on ESCI adds.

In [19]:
records, per_query = {}, {}


def split_part(split):
    return pairs.loc[MASK[split], ['example_id', 'query_id', 'query', 'text', 'esci_label']]


def log_run(name, scores, info, compare_to=None):
    # scores: {split: array in the row order of split_part(split)}
    rec = {'run': name, 'mode': MODE, **info, 'tie_seeds': list(TIE_SEEDS), 'bootstrap': 1000,
           'created_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime()), 'metrics': {}}
    per_query[name] = {}
    for split, sc in scores.items():
        part = split_part(split)
        rec['metrics'][split], per_query[name][split] = summarize(part, sc)
        part[['example_id', 'query_id']].assign(score=np.asarray(sc, dtype='float32')).to_parquet(
            RES / 'scores' / f'{name}_{split}.parquet', index=False)
    if compare_to in per_query:
        rec['paired_vs'] = {compare_to: {s: paired_bootstrap(per_query[compare_to][s], per_query[name][s])
                                         for s in scores if s in per_query[compare_to]}}
    records[name] = rec
    (RES / f'{name}.json').write_text(json.dumps(rec, indent=2, ensure_ascii=False))
    print(name, {s: (v['ndcg'], v['ndcg_ci95'], 'E>S', v['e_over_s']) for s, v in rec['metrics'].items()})
    return rec


def load_earlier_run(run):
    # Per-query metrics of a run saved by an earlier notebook (attached as input); False if not attached.
    for split in EVAL_SPLITS:
        hits = glob.glob(f'/kaggle/input/**/scores/{run}_{split}.parquet', recursive=True)
        if not hits:
            return False
        part = split_part(split)
        sc = part[['example_id']].merge(pd.read_parquet(hits[0])[['example_id', 'score']], on='example_id', how='left')
        assert sc['score'].notna().all(), f'{run} scores do not cover {split}'
        m, per_query.setdefault(run, {})[split] = summarize(part, sc['score'].to_numpy())
        print(f'{run} {split}: nDCG {m["ndcg"]}')
    return True


REF = 'b1_bm25_main' if load_earlier_run('b1_bm25_main') else None
print('paired comparison with', REF or 'nothing (bm25 notebook output not attached)')

b1_bm25_main val: nDCG 0.8227
paired comparison with b1_bm25_main


In [20]:
model0 = AutoModelForSequenceClassification.from_pretrained(CFG['model'], num_labels=1).float().to(DEVICE)
zs = {s: predict(model0, tok, split_part(s), CFG['max_len']) for s in EVAL_SPLITS}
log_run(('smoke_' if MODE == 'smoke' else '') + 'ce_minilm_zeroshot', zs,
        {'model': CFG['model'], 'fields': CFG['fields'], 'max_len': CFG['max_len'], 'fine_tuned': False}, compare_to=REF)
del model0
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

smoke_ce_minilm_zeroshot {'val': (0.833, [0.827, 0.8388], 'E>S', 0.6262)}


In [22]:
val_df = split_part('val')
b3_runs = []
for seed in SEEDS:
    name = f"{'smoke_' if MODE == 'smoke' else ''}b3_minilm_{CFG['fields']}_{CFG['loss']}_s{seed}"
    print(f'==== {name}')
    model, tokenizer, info = train_ce(CFG, train, seed, eval_df=val_df, eval_every=EVAL_EVERY)
    scores = {s: predict(model, tokenizer, split_part(s), CFG['max_len']) for s in EVAL_SPLITS}
    log_run(name, scores, {**CFG, 'seed': seed, 'subsample': SUBSAMPLE,
                           'train_queries': int(train['query_id'].nunique()), 'train_pairs': len(train),
                           'gpu': torch.cuda.get_device_name(0), **info}, compare_to=REF)
    b3_runs.append(name)
    if SAVE_MODEL:
        model.save_pretrained(OUT / f'model_{name}')
        tokenizer.save_pretrained(OUT / f'model_{name}')
    del model
    torch.cuda.empty_cache()

==== smoke_b3_minilm_title_mse_s42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_50/3272265784.py:81: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


step    100/583 | loss 36.11465 | 18.5 it/s | 0.1 min, ~0 min left
   val @ 100: nDCG 0.8342 | E-over-S 0.6277
step    200/583 | loss 8.00973 | 6.6 it/s | 0.5 min, ~1 min left
   val @ 200: nDCG 0.7801 | E-over-S 0.5636
step    300/583 | loss 0.27932 | 5.5 it/s | 0.9 min, ~1 min left
   val @ 300: nDCG 0.8189 | E-over-S 0.6107
step    400/583 | loss 0.22734 | 5.0 it/s | 1.3 min, ~1 min left
   val @ 400: nDCG 0.8265 | E-over-S 0.622
step    500/583 | loss 0.22094 | 4.8 it/s | 1.7 min, ~0 min left
   val @ 500: nDCG 0.8291 | E-over-S 0.6234
step    583/583 | loss 0.22163 | 4.5 it/s | 2.1 min, ~0 min left
   val @ 583: nDCG 0.8299 | E-over-S 0.6255
smoke_b3_minilm_title_mse_s42 {'val': (0.8299, [0.8236, 0.8362], 'E>S', 0.6255)}


In [23]:
if MODE == 'smoke':
    r = records[b3_runs[0]]
    zs_name = 'smoke_ce_minilm_zeroshot'
    first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
    checks = {
        'loss goes down': last < first,
        'val nDCG >= zero-shot': r['metrics']['val']['ndcg'] >= records[zs_name]['metrics']['val']['ndcg'],
        'val nDCG > BM25 title (0.816)': r['metrics']['val']['ndcg'] > 0.816,
    }
    print(f'loss {first:.5f} -> {last:.5f} | speed {r["it_per_s"]} it/s -> '
          f'full epoch ~{len(train_all) / CFG["batch_size"] / r["it_per_s"] / 60:.0f} min per seed')
    for k, v in checks.items():
        print(('OK    ' if v else 'CHECK ') + k)
    print('Smoke test passed: switch MODE to full.' if all(checks.values()) else
          'Some checks failed: send me this output before the full run.')

loss 36.11465 -> 0.22163 | speed 3.95 it/s -> full epoch ~50 min per seed
OK    loss goes down
CHECK val nDCG >= zero-shot
OK    val nDCG > BM25 title (0.816)
Some checks failed: send me this output before the full run.


## 5. Results

`ndcg_amazon` on test is the number to compare with Amazon's 0.857. Their training set was about 10% larger, so a gap of about 0.005 is not a failure to reproduce.

In [24]:
cols = ['ndcg', 'ndcg_ci95', 'ndcg_amazon', 'ndcg10', 'mrr_e', 'e_over_s']
table = pd.DataFrame([{'split': s, 'run': n, **{c: r['metrics'][s][c] for c in cols}}
                      for n, r in records.items() for s in r['metrics']]).set_index(['split', 'run']).sort_index()
display(table)

if len(b3_runs) > 1:
    for s in EVAL_SPLITS:
        v = {m: [records[n]['metrics'][s][m] for n in b3_runs] for m in ['ndcg', 'ndcg_amazon', 'e_over_s']}
        print(s, {m: f'mean {np.mean(x):.4f} (seeds {x})' for m, x in v.items()})
    print('Amazon reference (test, amazon_script gains): 0.857')

for n, r in records.items():
    for ref, by_split in r.get('paired_vs', {}).items():
        for s, v in by_split.items():
            print(f'{s:4s} {n} - {ref}: {v}')

ndcg         ndcg_ci95  ndcg_amazon  \
split run                                                                    
val   smoke_b3_minilm_title_mse_s42  0.8299  [0.8236, 0.8362]       0.8081   
      smoke_ce_minilm_zeroshot       0.8330   [0.827, 0.8388]       0.8124   

                                     ndcg10   mrr_e  e_over_s  
split run                                                      
val   smoke_b3_minilm_title_mse_s42  0.7102  0.7801    0.6255  
      smoke_ce_minilm_zeroshot       0.7163  0.7868    0.6262

val  smoke_ce_minilm_zeroshot - b1_bm25_main: {'diff': 0.0103, 'ci95': [0.0055, 0.0156], 'p_b_le_a': 0.0}
val  smoke_b3_minilm_title_mse_s42 - b1_bm25_main: {'diff': 0.0071, 'ci95': [0.0017, 0.0129], 'p_b_le_a': 0.007}


## 6. GPU benchmark (smoke mode only)

Times 100 training steps (after 20 warm-up steps) for the configurations planned in Direction 1, using real `main` or `title` text. `hours_per_epoch` is for the whole train split (378,140 pairs, batch 32). The numbers decide `max_len` for DeBERTa and whether out-of-fold training for Direction 3 uses 5 or 2 folds (rule: 5 folds if one DeBERTa epoch takes at most about 8 GPU hours).

In [26]:
def benchmark(model_name, fields, max_len, batch_size=32, n_steps=100, warm=20):
    smp = pairs[pairs['split'] == 'train'].sample(batch_size * (n_steps + warm), random_state=SEED)
    assert fields in (CFG['fields'], 'main')
    q = smp['query'].to_numpy()
    t = (smp['text'] if fields == CFG['fields'] else smp['product_id'].map(MAIN_TEXT)).to_numpy()
    y = torch.from_numpy(smp['gain'].to_numpy(np.float32)).to(DEVICE)
    row = {'model': model_name, 'fields': fields, 'max_len': max_len, 'batch_size': batch_size}
    model = opt = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).float().to(DEVICE)
        cfg = {**CFG, 'model': model_name}
        opt, scaler = make_optimizer(model, cfg), make_scaler(True)
        torch.cuda.reset_peak_memory_stats()
        finite = True
        for i in range(warm + n_steps):
            if i == warm:
                torch.cuda.synchronize()
                t0 = time.time()
            sl = slice(i * batch_size, (i + 1) * batch_size)
            with torch.autocast('cuda', dtype=torch.float16):
                pred = model(**encode(tokenizer, q[sl], t[sl], max_len)).logits.squeeze(-1)
            loss = F.mse_loss(pred.float(), y[sl])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            finite &= bool(torch.isfinite(loss))
        torch.cuda.synchronize()
        sec = (time.time() - t0) / n_steps
        row.update(sec_per_step=round(sec, 4), hours_per_epoch=round(sec * len(train_all) / batch_size / 3600, 2),
                   peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 1), loss_finite=finite)
    except torch.cuda.OutOfMemoryError:
        row['error'] = 'out of memory'
    finally:
        del model, opt
        gc.collect()
        torch.cuda.empty_cache()
    print(row)
    return row


if BENCHMARK:
    bench = [benchmark(m, f, L) for m, f, L in [
        ('cross-encoder/ms-marco-MiniLM-L-12-v2', 'title', 128),
        ('cross-encoder/ms-marco-MiniLM-L-12-v2', 'main', 256),
        ('microsoft/deberta-v3-base', 'title', 128),
        ('microsoft/deberta-v3-base', 'main', 192),
        ('microsoft/deberta-v3-base', 'main', 256),
    ]]
    bench_df = pd.DataFrame(bench)
    display(bench_df)
    (RES / 'gpu_benchmark.json').write_text(json.dumps(
        {'gpu': torch.cuda.get_device_name(0), 'created_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime()),
         'rows': bench}, indent=2))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'title', 'max_len': 128, 'batch_size': 32, 'sec_per_step': 0.0578, 'hours_per_epoch': 0.19, 'peak_gb': 2.1, 'loss_finite': True}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'main', 'max_len': 256, 'batch_size': 32, 'sec_per_step': 0.1661, 'hours_per_epoch': 0.55, 'peak_gb': 3.0, 'loss_finite': True}


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

{'model': 'microsoft/deberta-v3-base', 'fields': 'title', 'max_len': 128, 'batch_size': 32, 'sec_per_step': 0.2221, 'hours_per_epoch': 0.73, 'peak_gb': 6.8, 'loss_finite': True}


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

{'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 192, 'batch_size': 32, 'sec_per_step': 0.5347, 'hours_per_epoch': 1.75, 'peak_gb': 10.2, 'loss_finite': True}


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

{'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256, 'batch_size': 32, 'sec_per_step': 0.7131, 'hours_per_epoch': 2.34, 'peak_gb': 12.5, 'loss_finite': True}


,model,fields,max_len,batch_size,sec_per_step,hours_per_epoch,peak_gb,loss_finite
0,cross-encoder/ms-marco-MiniLM-L-12-v2,title,128,32,0.0578,0.19,2.1,True
1,cross-encoder/ms-marco-MiniLM-L-12-v2,main,256,32,0.1661,0.55,3.0,True
2,microsoft/deberta-v3-base,title,128,32,0.2221,0.73,6.8,True
3,microsoft/deberta-v3-base,main,192,32,0.5347,1.75,10.2,True
4,microsoft/deberta-v3-base,main,256,32,0.7131,2.34,12.5,True
